# Unit 1: PyTorch 基础

## 学习目标
- 理解 PyTorch Tensor 的概念与 NumPy 的关系
- 掌握 Tensor 的创建、操作和变形
- 学会使用 GPU 加速计算
- 理解 Autograd 自动求导机制
- 理解动态计算图的基本概念

PyTorch 是目前最流行的深度学习框架之一，由 Meta AI 开发维护。它的核心设计理念是**动态计算图**和**Pythonic API**，让研究人员和工程师能够快速迭代实验。

## 1.1 Tensor 基础

Tensor（张量）是 PyTorch 的核心数据结构，类似于 NumPy 的 ndarray，但额外支持 GPU 加速和自动求导。

**标量 → 向量 → 矩阵 → 张量**的层级关系：
- Scalar: 0 维张量，如 `3.14`
- Vector: 1 维张量，如 `[1, 2, 3]`
- Matrix: 2 维张量，如 `[[1,2],[3,4]]`
- Tensor: 3+ 维张量，广泛用于批量数据处理

In [ ]:
import torch
import numpy as np

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS (Apple Silicon) available: {torch.backends.mps.is_available()}")

In [ ]:
# 从列表创建 Tensor
a = torch.tensor([1, 2, 3])
print(f"从列表创建: {a}")
print(f"dtype: {a.dtype}, shape: {a.shape}, device: {a.device}\n")

# 从 NumPy 创建
b = torch.from_numpy(np.array([4.0, 5.0, 6.0]))
print(f"从 NumPy 创建: {b}")

# 使用工厂方法
zeros = torch.zeros(2, 3)        # 全零
ones = torch.ones(2, 3)          # 全一
randn = torch.randn(2, 3)        # 标准正态分布
arange = torch.arange(0, 10, 2)  # 等差数列

print(f"zeros shape {zeros.shape}:\n{zeros}\n")
print(f"randn shape {randn.shape}:\n{randn}")

In [ ]:
# dtype 类型转换
x = torch.tensor([1, 2, 3], dtype=torch.float32)
x_long = x.long()       # 转 int64
x_half = x.half()       # 转 float16
print(f"float32: {x.dtype}")
print(f"long:    {x_long.dtype}")
print(f"half:    {x_half.dtype}")

## 1.2 Tensor 操作

PyTorch 提供了丰富的 Tensor 操作，包括数学运算、索引切片、变形等。

In [ ]:
# 基本数学运算
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print(f"a + b   = {a + b}")          # 逐元素加法
print(f"a * b   = {a * b}")          # 逐元素乘法
print(f"a @ b   = {a @ b}")          # 点积 (dot product)
print(f"a ** 2  = {a ** 2}")         # 幂运算

In [ ]:
# 矩阵乘法
A = torch.randn(3, 4)
B = torch.randn(4, 2)
C = A @ B                          # 等价于 torch.matmul(A, B)
print(f"A(3x4) @ B(4x2) = C(3x2):\n{C}")
print(f"shape: {C.shape}")

In [ ]:
# Broadcasting 广播机制
# 当两个张量形状不同时，PyTorch 会自动扩展维度使其兼容
x = torch.ones(3, 1)
y = torch.ones(4)
z = x + y
print(f"x shape: {x.shape}")
print(f"y shape: {y.shape}")
print(f"z shape (broadcasted): {z.shape}")
print(f"z:\n{z}")

r.view(3, 4) 之所以成功，是因为 reshape 返回的 r 恰好仍然是内存连续的。  
这引出了理解 view vs reshape 最关键的一个误区：很多人以为 "reshape = 不安全时的复制"，  
但实际上 reshape 优先返回视图（共享内存），只有在无法用视图表达时才复制。

In [ ]:
# 变形操作: view vs reshape
t = torch.arange(12)
print(f"原始: {t}")

# view 要求内存连续，返回共享内存的视图
v = t.view(3, 4)
print(f"view(3,4):\n{v}")

# reshape 优先返回视图，必要时才会复制
r = t.reshape(4, 3)
print(f"reshape(4,3):\n{r}")

s = r.view(3, 4)
print(f"view(3,4):\n{s}")



# 常用变形: squeeze 去掉维度为1的轴, unsqueeze 增加维度
q = torch.randn(1, 3, 1, 4)
print(f"\nsqueeze 前: {q.shape}")
print(f"squeeze 后: {q.squeeze().shape}")                # 去掉所有大小为1的维度: (3, 4)
print(f"squeeze(dim=0): {q.squeeze(0).shape}")           # (3, 1, 4)
print(f"unsqueeze(dim=0): {r.unsqueeze(0).shape}")       # (1, 4, 3)

In [ ]:
# 拼接与堆叠
a = torch.tensor([[1, 2], [3, 4]])
b = torch.tensor([[5, 6], [7, 8]])

cat_dim0 = torch.cat([a, b], dim=0)    # 沿行拼接 (4, 2)
cat_dim1 = torch.cat([a, b], dim=1)    # 沿列拼接 (2, 4)
stacked  = torch.stack([a, b], dim=0)  # 新维度堆叠 (2, 2, 2)

print(f"cat(dim=0) shape {cat_dim0.shape}:\n{cat_dim0}\n")
print(f"cat(dim=1) shape {cat_dim1.shape}:\n{cat_dim1}\n")
print(f"stack(dim=0) shape {stacked.shape}:\n{stacked}")

## 1.3 设备管理: CPU / GPU / MPS

PyTorch 支持多种计算设备。在 Apple Silicon Mac 上可以使用 MPS 加速。

In [ ]:
# 检测可用设备并选择最佳设备
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

In [ ]:
# 将 Tensor 移动到指定设备
x = torch.randn(1000, 1000)
x_device = x.to(device)
print(f"x:       {x.device}")
print(f"x_device: {x_device.device}")

# 设备间运算需要相同设备
# y = x + x_device  会报错！
y = x.to(device) + x_device  # 正确
print(f"y: {y.device}")


In [ ]:
# CPU <-> GPU/设备 数据传输性能对比
import time

if device.type != "cpu":
    # GPU 上的矩阵乘法
    a_gpu = torch.randn(2000, 2000, device=device)
    b_gpu = torch.randn(2000, 2000, device=device)
    _ = torch.mm(a_gpu, b_gpu)  # warm up
    
    start = time.time()
    for _ in range(10):
        torch.mm(a_gpu, b_gpu)
    if device.type == "cuda":
        # 确保所有 CUDA 操作完成后再计时
        torch.cuda.synchronize()
    gpu_time = time.time() - start
    print(f"GPU 矩阵乘法 10次: {gpu_time:.4f}s")

    # CPU 对比
    a_cpu = a_gpu.cpu()
    b_cpu = b_gpu.cpu()
    _ = torch.mm(a_cpu, b_cpu)
    start = time.time()
    for _ in range(100):
        torch.mm(a_cpu, b_cpu)
    cpu_time = time.time() - start
    print(f"CPU 矩阵乘法 10次: {cpu_time:.4f}s")
    print(f"加速比: {cpu_time / gpu_time:.1f}x")
else:
    print("当前为 CPU 设备，跳过性能对比")

## 1.4 Autograd: 自动求导

Autograd 是 PyTorch 的自动微分引擎。它记录所有对 Tensor 的操作，构建计算图，然后自动计算梯度。

**核心步骤：**
1. 设置 `requires_grad=True` 标记需要追踪的 Tensor
2. 执行前向计算
3. 调用 `.backward()` 自动计算梯度
4. 通过 `.grad` 属性获取梯度

In [ ]:
# 简单示例: y = x^2, dy/dx = 2x
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2

print(f"x = {x.item()}")
print(f"y = {y.item()}")
print(f"requires_grad: x={x.requires_grad}, y={y.requires_grad}")
print(f"grad_fn of y: {y.grad_fn}")  # y 的梯度函数

# 反向传播
y.backward()
print(f"dy/dx = x.grad = {x.grad.item()}")  # 应为 2*3 = 6

In [ ]:
# 多元函数梯度
# z = x^2 + y^3, 求 ∂z/∂x 在 (x=2, y=3) 处的值
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)
z = x**2 + y**3

z.backward()
print(f"∂z/∂x = {x.grad.item()}")  # 2*2 = 4
print(f"∂z/∂y = {y.grad.item()}")  # 3*3^2 = 27

In [ ]:
# 梯度累积与清零
# 每次 .backward() 会累加梯度，需要手动清零
w = torch.tensor([1.0, 2.0], requires_grad=True)

for i in range(3):
    loss = (w ** 2).sum()
    loss.backward()
    print(f"Step {i}: w.grad = {w.grad}")
    # w.grad.zero_()  # 如果不清零，梯度会累加

In [ ]:
# .detach() 和 torch.no_grad() 的区别
x = torch.tensor([1.0, 2.0], requires_grad=True)

# no_grad: 上下文中的所有计算都不会被追踪（推理时使用）
with torch.no_grad():
    y = x * 2  # y 没有 grad_fn
    print(f"no_grad 中 y.requires_grad: {y.requires_grad}")

# detach: 创建一个不需要梯度的副本，但保持原 Tensor 不变
z = x.detach() * 3
print(f"detach 后 z.requires_grad: {z.requires_grad}")
print(f"原 x.requires_grad: {x.requires_grad}")

## 1.5 动态计算图

PyTorch 使用**动态计算图**（Define-by-Run）：
- 图结构在每次前向传播时动态构建
- 支持 Python 原生控制流（if/for/while）
- 每次迭代可以有不同的图结构

这与 TensorFlow 1.x 的**静态图**（Define-and-Run）形成对比。

In [ ]:
# 动态图：每次迭代可以使用不同的计算路径
def dynamic_forward(x, threshold=0):
    if x.sum() > threshold:
        return x * 2
    else:
        return x * 3

x1 = torch.tensor([1.0, 2.0], requires_grad=True)
x2 = torch.tensor([-3.0, 1.0], requires_grad=True)

y1 = dynamic_forward(x1)
y2 = dynamic_forward(x2)

print(f"x1.sum() = {x1.sum().item()}, y1 = {y1}, grad_fn: {y1.grad_fn}")
print(f"x2.sum() = {x2.sum().item()}, y2 = {y2}, grad_fn: {y2.grad_fn}")

# 反向传播 y1
y1.sum().backward()
print("x1.grad:", x1.grad)  # tensor([2., 2.])

# 反向传播 y2（需要重新创建或清零梯度）
x2.grad = None  # 清零
y2.sum().backward()
print("x2.grad:", x2.grad)  # tensor([3., 3.])

## 1.6 与 NumPy 的互操作

PyTorch 和 NumPy 之间可以轻松转换，共享底层内存（CPU 上）。

In [ ]:
# Tensor -> NumPy
t = torch.ones(5)
n = t.numpy()
print(f"Tensor: {t}")
print(f"NumPy:  {n}")

# 共享内存：修改 Tensor 会影响 NumPy
t.add_(1)  # in-place 加法
print(f"修改 Tensor 后 NumPy: {n}")

In [ ]:
# NumPy -> Tensor
n = np.array([2.0, 4.0, 6.0])
t1 = torch.from_numpy(n)   # 共享内存
t2 = torch.tensor(n)       # 复制数据

n[0] = 99
print(f"修改 NumPy 后 from_numpy: {t1}")  # 变了
print(f"修改 NumPy 后 tensor():    {t2}")  # 不变

## 要点总结

| 概念 | 关键点 |
|------|--------|
| Tensor | 类似 NumPy ndarray，支持 GPU + 自动求导 |
| view vs reshape | view 要求连续内存（共享），reshape 自动处理 |
| Device | `.to(device)` 切换设备，同设备才能运算 |
| Autograd | `requires_grad=True` → `.backward()` → `.grad` |
| 动态图 | 每次前向传播构建新图，支持 Python 控制流 |
| 梯度清零 | 每次 backward 前需要 `.zero_()` |

### 练习
1. 创建一个 3x4 的随机张量，计算每行的平均值
2. 实现 `f(x) = x^3 + 2x^2` 在 `x=4` 处的梯度
3. 尝试将同一个 Tensor 在 CPU 和 GPU 间来回移动

In [ ]:
# 练习参考代码

# 1. 创建 3x4 随机张量，计算每行平均值
t = torch.randn(3, 4)
row_mean = t.mean(dim=1)
print(f"Tensor:\n{t}")
print(f"1.每行均值: {row_mean}")

# 2. f(x) = x^3 + 2x^2 在 x=4 处的梯度
x = torch.tensor(4.0, requires_grad=True)
f = x**3 + 2 * x**2
f.backward()
print(f"2.f'(4) = {x.grad.item()}")  # 3*16 + 4*4 = 64